# Multiclass Classification

CSCI 6379 · Topic 12. Handle c > 2 classes by computing ONE linear score per class and predicting the argmax. Labels are one-hot encoded; each class column is trained by the same MSE gradient descent (one-vs-all).

In [ ]:
import numpy as np

# 3 classes, 4 features, 15 training samples (5 per class).
trainX = np.array([[0.1, 0.2, 0.3, 0.2], [0.5, 0.4, 0.3, 0.7], [0.3, 0.7, 0.4, 0.1],
                   [0.2, 0.8, 0.9, 0.3], [1.1, 0.5, 0.2, 0.9],
                   [4.3, 5.3, 4.7, 4.2], [4.5, 5.1, 5.3, 4.4], [5.1, 4.8, 5.1, 4.6],
                   [4.9, 4.6, 4.9, 4.3], [5.4, 5.5, 4.3, 4.7],
                   [10.1, 10.2, 10.3, 11.3], [11.3, 11.2, 11.1, 10.3], [12.5, 12.3, 12.1, 11.4],
                   [11.7, 11.8, 11.2, 12.8], [13.1, 10.2, 12.4, 11.7]])

# one-hot labels: class 0 -> [1,0,0], class 1 -> [0,1,0], class 2 -> [0,0,1]
trainY = np.array([[1,0,0]]*5 + [[0,1,0]]*5 + [[0,0,1]]*5)

testX = np.array([[0.5, 0.4, 0.6, 0.5], [5.4, 5.6, 5.5, 5.2], [11.7, 11.6, 11.5, 11.4]])
testY = np.array([[1,0,0], [0,1,0], [0,0,1]])

## One-hot encoding

`np.eye(c)[labels]` builds one-hot rows from a plain label vector — handy for any dataset.

In [ ]:
labels = np.array([0, 1, 2, 0, 2])
print(np.eye(3, dtype=int)[labels])
# [[1 0 0]
#  [0 1 0]
#  [0 0 1]
#  [1 0 0]
#  [0 0 1]]

## Train one linear score per class

W has one COLUMN per class (shape m x c). Each column is trained with the SAME MSE gradient descent as linear regression, aimed at its own one-hot column.

In [ ]:
W = np.zeros((4, 3))     # m x c : one weight column per class
B = np.zeros(3)
alpha = 0.005
N = trainX.shape[0]
c = W.shape[1]

for _ in range(1000):
    for j in range(c):                       # train each class's score
        err = np.dot(trainX, W[:, j]) + B[j] - trainY[:, j]
        W[:, j] = W[:, j] - alpha * (1/N) * np.dot(err, trainX)
        B[j]    = B[j]    - alpha * (1/N) * np.sum(err)

print("W =\n", W.round(4))
print("B =", B.round(4))

## Predict by argmax, then measure accuracy

Each test row produces 3 scores; the predicted class is the index of the largest.

In [ ]:
scores = np.dot(testX, W) + B
print("test scores:\n", scores.round(3))
pred = np.argmax(scores, axis=1)
print("pred:", pred, " true:", np.argmax(testY, axis=1))
print("accuracy:", np.mean(pred == np.argmax(testY, axis=1)))   # -> 1.0

## The raw scores are not probabilities — softmax preview

Softmax squashes any score vector into a probability distribution (sums to 1). We train that with cross-entropy in the neural-network topic; here it is just a preview of the first test row.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max())
    return e / e.sum()

print("scores :", scores[0].round(3))
print("softmax:", softmax(scores[0]).round(3), " (sums to", round(softmax(scores[0]).sum(), 3), ")")

## Assignment: Multiclass Classification on the Iris Dataset

Apply this topic's classifier to the **Iris** dataset (150 samples, 4 features, 3 classes). Shuffle, split 80/20, and — inside the training loop — store the error (1 - accuracy) on both the train and test sets each iteration. Then plot train error (blue) vs test error (red). Below is a working starter; extend it with the plot and your analysis.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)                 # 150 samples, 4 features, 3 classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42)
Y_train = np.eye(3)[y_train]                       # one-hot encode

W = np.zeros((4, 3)); B = np.zeros(3)
alpha = 0.01; N = len(X_train); c = 3
train_err, test_err = [], []
for it in range(3000):
    for j in range(c):                             # same per-column GD as this topic
        err = X_train @ W[:, j] + B[j] - Y_train[:, j]
        W[:, j] -= alpha * (1/N) * (err @ X_train)
        B[j]    -= alpha * (1/N) * err.sum()
    train_err.append(1 - np.mean(np.argmax(X_train @ W + B, 1) == y_train))
    test_err.append( 1 - np.mean(np.argmax(X_test  @ W + B, 1) == y_test))

print("final train acc %.3f  test acc %.3f" % (1-train_err[-1], 1-test_err[-1]))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_err, 'b', label="train error")
plt.plot(test_err, 'r', label="test error")
plt.xlabel("iteration"); plt.ylabel("error (1 - accuracy)"); plt.legend(); plt.show()

# TODO for the assignment:
#   - analyze: do both errors fall? do train and test diverge (overfitting)?
#   - what final accuracy do you reach?  (hint: features are on different
#     scales -- would normalization from Topic 9 help convergence?)